# 17 — Ponto de entrada `python -m app`

Desenvolve `_dados_demo` e `main`. **NF5.**

In [ ]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np
import pandas as pd
import os
from app.principal import executar_pipeline

setup OK


## Desenvolvimento

As funções abaixo foram escritas aqui e, após os testes, movidas para `app/__main__.py`.

In [ ]:
def _dados_demo(T: int = 300, seed: int = 7) -> pd.DataFrame:
    """Série sintética de retornos mensais (Ibovespa + CDI) para a demo."""
    rng = np.random.default_rng(seed)
    ruido = rng.normal(0.0, 0.06, T)
    ruido -= ruido.mean()                       # média exatamente 0
    return pd.DataFrame({
        "data": pd.date_range("2000-01", periods=T, freq="MS").strftime("%Y-%m"),
        "ibov": 0.015 + ruido,                  # prêmio positivo ⇒ α* interior
        "cdi": np.full(T, 0.008),
    })


def main() -> None:
    db = os.path.join("data", "mercado.db")
    comum = {"ativos": ["ibov"], "gamma": 5.0, "beta": 0.96, "w0": 1.0,
             "horizonte": 60, "n_scenarios": 60_000, "n_paths": 3_000, "seed": 1}
    if os.path.exists(db):
        print(f"(dados REAIS: {db} - R_f vem da serie real do CDI)")
        config = {"db_path": db, "tabela": "retornos", **comum}
    else:
        print("(SEM banco real -> dados SINTETICOS de demonstracao)")
        print("  para baixar dados reais:  python -m app.ingestao")
        config = {"retornos": _dados_demo(), "cdi_anual": 0.10, **comum}
    res = executar_pipeline(config)

    print("=== Esteira DP-CRRA-IID (Samuelson 1969) ===")
    print(f"Ativos de risco      : {res['ativos']}")
    print(f"R_f (mensal)         : {res['rf']:.6f}")
    print(f"mu_hat               : {np.round(res['mu_hat'], 5)}")
    print(f"Carteira otima a*   : {np.round(res['alpha_star'], 4)}")
    print(f"Phi_hat              : {res['phi_hat']:.6f}")
    print(f"Consumo inicial c_0  : {res['consumo_inicial']:.4f}  (theta_0={res['theta'][0]:.4f})")
    print(f"theta_T (terminal)   : {res['theta'][-1]:.4f}")
    print(f"E[W_T] (T={res['horizonte']})        : {res['E_W_T']:.4f}  "
          f"[P5={res['W_T_p5']:.4f}, P95={res['W_T_p95']:.4f}]")


**Teste** — `main()` roda a esteira e imprime o resultado.

In [3]:
import io, contextlib
print('_dados_demo().head():'); print(_dados_demo().head())
os.chdir(RAIZ)
buf = io.StringIO()
with contextlib.redirect_stdout(buf): main()
print(buf.getvalue())
assert 'Carteira otima' in buf.getvalue()
print('entrypoint: PASSOU')

_dados_demo().head():
      data      ibov    cdi
0  2000-01  0.022986  0.008
1  2000-02  0.040837  0.008
2  2000-03  0.006464  0.008
3  2000-04 -0.030524  0.008
4  2000-05 -0.004368  0.008
(dados REAIS: data\mercado.db - R_f vem da serie real do CDI)
=== Esteira DP-CRRA-IID (Samuelson 1969) ===
Ativos de risco      : ['ibov']
R_f (mensal)         : 0.009564
mu_hat               : [0.00971]
Carteira otima a*   : [-0.0193]
Phi_hat              : 0.962625
Consumo inicial c_0  : 0.0253  (theta_0=0.0253)
theta_T (terminal)   : 1.0000
E[W_T] (T=60)        : 0.0174  [P5=0.0171, P95=0.0177]

entrypoint: PASSOU
